# Semana 5 · ML clásico en síntesis

**Nivel Intermedio** · Semana 5 · Clase expositiva [A] · 2 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/clases/sem05_ml_clasico.ipynb)

**Objetivos.** Al terminar la clase podrás:

1. Entrenar y evaluar cuatro modelos clásicos con el patrón `fit/predict` de sklearn: regresión lineal, regresión logística, kNN y árbol de decisión.
2. Separar los datos en entrenamiento, validación y prueba, y explicar para qué sirve cada parte.
3. Reconocer el sobreajuste (*overfitting*) y el subajuste en una curva de error, explicarlos con sesgo y varianza, y elegir un hiperparámetro con validación cruzada (*cross-validation*).
4. Calcular e interpretar *accuracy*, precisión, *recall* y F1, y decidir cuál importa en un problema.

**Prerrequisitos.** Semanas 2 a 4: NumPy, pandas y la idea de entrenar como minimizar una pérdida. En el B0 ya usaste `LogisticRegression` y `DummyClassifier`; hoy vemos qué hay detrás. Lectura previa: *Supervised Learning*, *Inference, Training, Test and Evaluation* y *Generalisation, Overfitting and Bias* de MÓLÓ.

**Agenda** (después de la apertura de 15 minutos).

| Parte | Tema | Minutos |
|---|---|---|
| 1 | Aprendizaje supervisado y el patrón `fit/predict` | 10 |
| 2 | Entrenamiento, validación y prueba | 15 |
| 3 | Sobreajuste, sesgo y varianza | 20 |
| 4 | Validación cruzada | 15 |
| 5 | Métricas: *accuracy*, precisión, *recall*, F1 | 20 |
| 6 | Los cuatro modelos, lado a lado | 15 |
| 7 | La receta completa (la del Sprint 0) | 5 |
| 8 | Cierre y tareas para la semana 6 | 5 |

Ejecuta esta celda primero: carga las librerías y los datos, fija la semilla y define `check`, la función que revisa tus ejercicios de "Tu turno".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 0
rng = np.random.default_rng(SEED)

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
COLS = ["bill_length_mm", "flipper_length_mm", "body_mass_g", "species"]
penguins = pd.read_csv(URL).dropna(subset=COLS)


def check(name, test):
    """Ejecuta una prueba y muestra ✓, ✗ o "todavía sin resolver", sin detener el notebook."""
    try:
        test()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"… {name}: todavía sin resolver")
    except AssertionError as e:
        print(f"✗ {name}: {e}")
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__}: {e}")


print(len(penguins), "pingüinos")

## 1. Aprendizaje supervisado y el patrón `fit/predict`

En el **aprendizaje supervisado** (*supervised learning*) tenemos ejemplos con su respuesta: las entradas $X$ (una fila por ejemplo, una columna por variable) y las etiquetas $y$. El modelo aprende de esos ejemplos una función que da la respuesta para entradas nuevas. Es como aprender a manejar con un instructor al lado que te corrige: después tienes que manejar solo.

Hay dos tipos de problemas, según la respuesta:

- **Regresión**: la respuesta es un número. Ejemplo: la masa de un pingüino a partir del largo de su aleta.
- **Clasificación**: la respuesta es una categoría. Ejemplo: la especie de un pingüino.

En sklearn todos los modelos se usan igual, con dos métodos:

- `model.fit(X, y)`: **entrenar**, es decir, buscar los parámetros que minimizan la pérdida (lo que hicimos a mano en la semana 4).
- `model.predict(X_new)`: **predecir** con los parámetros ya aprendidos.

`X` siempre es bidimensional, con forma `(n, d)`, aunque tenga una sola columna; por eso usamos `penguins[["flipper_length_mm"]]`, con doble corchete. La regresión lineal de la semana 4, en tres líneas:

In [ ]:
reg = LinearRegression()
reg.fit(penguins[["flipper_length_mm"]], penguins["body_mass_g"])
print(f"w = {reg.coef_[0]:.1f} gramos por milímetro de aleta,  b = {reg.intercept_:.0f} gramos")

new = pd.DataFrame({"flipper_length_mm": [180, 200, 220]})
for mm, grams in zip(new["flipper_length_mm"], reg.predict(new)):
    print(f"aleta de {mm} mm → masa predicha {grams:.0f} g")

Es la misma recta de la semana 4 (allá con la aleta estandarizada y la masa en kilos): sklearn encontró el mínimo del MSE con una fórmula en lugar de dar pasos de gradiente.

Para clasificar, la **regresión logística** (*logistic regression*) es exactamente el clasificador *softmax* que entrenamos a mano al final de la semana 4: una capa lineal, *softmax* y entropía cruzada. Usamos las mismas dos medidas para separar las tres especies, y `score` da la *accuracy*, la fracción de aciertos. Por ahora la medimos sobre los mismos datos con los que entrenó; en la parte 2 veremos por qué eso es hacer trampa.

In [ ]:
X_peng = penguins[["bill_length_mm", "flipper_length_mm"]].to_numpy()
y_peng = penguins["species"].to_numpy()

clf = make_pipeline(StandardScaler(), LogisticRegression())
clf.fit(X_peng, y_peng)
print("predicciones para los 5 primeros:", clf.predict(X_peng[:5]))
print("probabilidades del primero:      ", clf.predict_proba(X_peng[:1]).round(3), "←", clf.classes_)
print(f"accuracy sobre los datos de entrenamiento: {clf.score(X_peng, y_peng):.3f}")

`make_pipeline(StandardScaler(), LogisticRegression())` encadena dos pasos: primero estandariza cada columna (resta la media y divide por la desviación estándar, como en la semana 4) y después ajusta el modelo. Todo el *pipeline* se usa como un solo modelo, con `fit` y `predict`.

### Tu turno 1: tu primer kNN

El modelo **kNN** (*k nearest neighbors*, los $k$ vecinos más cercanos) no aprende parámetros: guarda todos los ejemplos y, para un pingüino nuevo, busca los $k$ más parecidos y vota. "Dime con quién andas y te diré quién eres".

Con los datos de entrenamiento y prueba de abajo (la partición se explica en la parte 2):

1. Crea `knn`, un `KNeighborsClassifier` con `n_neighbors=5`, dentro de un *pipeline* con `StandardScaler()`.
2. Entrénalo con `X_tr_peng, y_tr_peng`.
3. Guarda en `acc_knn` su *accuracy* sobre `X_te_peng, y_te_peng`.

In [ ]:
X_tr_peng, X_te_peng, y_tr_peng, y_te_peng = train_test_split(
    X_peng, y_peng, test_size=0.25, random_state=SEED, stratify=y_peng)

# TODO: crea knn, entrénalo y calcula acc_knn
knn = None
acc_knn = None

In [ ]:
def test_knn():
    if knn is None or acc_knn is None:
        raise NotImplementedError
    assert knn[-1].n_neighbors == 5, "usa n_neighbors=5"
    assert hasattr(knn[-1], "classes_"), "falta entrenar el modelo con fit"
    assert np.isclose(acc_knn, knn.score(X_te_peng, y_te_peng)), "acc_knn debe medirse sobre los datos de prueba"


check("kNN de los pingüinos", test_knn)

## 2. Entrenamiento, validación y prueba

Si un profesor toma el examen con las mismas preguntas que resolvió en clase, un estudiante que se memorizó las respuestas saca 10 sin haber entendido nada. Con un modelo pasa lo mismo: **medirlo con los datos con los que entrenó no dice si aprendió o si memorizó**.

Lo vemos con un **árbol de decisión** (*decision tree*), un modelo que clasifica con una cadena de preguntas de sí o no, como el juego de las 20 preguntas ("¿el radio del tumor mide más de 16?"). Sin límite de preguntas, el árbol puede seguir partiendo hasta que cada hoja tenga un solo ejemplo: se aprende los datos de memoria.

Usamos los datos de cáncer de mama del B0: 569 tumores con 30 medidas cada uno, y la etiqueta 1 si el tumor es **maligno** (lo que queremos detectar) y 0 si es benigno.

In [ ]:
cancer = load_breast_cancer()
X_bc = cancer.data
y_bc = 1 - cancer.target          # en sklearn 0 es maligno; lo damos vuelta para que 1 = maligno
print("forma de X:", X_bc.shape, "  fracción de malignos:", y_bc.mean().round(3))

X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(X_bc, y_bc, test_size=0.25, random_state=SEED, stratify=y_bc)
tree = DecisionTreeClassifier(random_state=SEED).fit(X_tr_bc, y_tr_bc)
print(f"árbol sin límite: accuracy de entrenamiento {tree.score(X_tr_bc, y_tr_bc):.3f}, "
      f"de prueba {tree.score(X_te_bc, y_te_bc):.3f}, profundidad {tree.get_depth()}")

Un 100% en entrenamiento y menos en datos que nunca vio. La cifra honesta es la segunda: lo que nos importa es la **generalización**, qué tan bien funciona el modelo con datos nuevos.

Por eso los datos se separan en tres partes, cada una con un papel:

| Parte | Para qué | Analogía |
|---|---|---|
| **Entrenamiento** (*train*) | `fit`: ajustar los parámetros | los ejercicios de la clase |
| **Validación** (*validation*) | comparar modelos y elegir hiperparámetros | los simulacros |
| **Prueba** (*test*) | la nota final, **una sola vez** al final | el examen, guardado en una caja fuerte |

Un **hiperparámetro** (*hyperparameter*) es una perilla que eliges tú antes de entrenar, no algo que aprende `fit`: el $k$ del kNN, la profundidad máxima del árbol, la tasa de aprendizaje de la semana 4. Si eliges las perillas mirando la prueba, la prueba deja de ser un examen sorpresa, y la nota sale inflada.

Dos detalles importantes al separar:

- **Mezclar antes de cortar.** Si los datos vienen ordenados (por especie, por fecha), cortar sin mezclar deja clases enteras fuera del entrenamiento.
- **Estratificar** (*stratify*): que cada parte tenga la misma proporción de clases que el total. Con `train_test_split(..., stratify=y)`.

### Tu turno 2: separar a mano

Completa `split_indices(n, val_frac, test_frac, seed)`. Debe mezclar los índices `0, …, n-1` con `np.random.default_rng(seed).permutation(n)` y devolver tres arreglos `(train_idx, val_idx, test_idx)`: los primeros `round(test_frac * n)` índices mezclados para la prueba, los siguientes `round(val_frac * n)` para la validación y el resto para el entrenamiento.

In [ ]:
def split_indices(n, val_frac, test_frac, seed):
    # TODO
    raise NotImplementedError

In [ ]:
def test_split_indices():
    tr, va, te = split_indices(100, 0.2, 0.1, seed=3)
    assert (len(tr), len(va), len(te)) == (70, 20, 10), f"tamaños {len(tr), len(va), len(te)}, se esperaba (70, 20, 10)"
    joined = np.concatenate([tr, va, te])
    assert len(set(joined)) == 100, "las tres partes no deben compartir índices"
    assert sorted(joined) == list(range(100)), "entre las tres deben estar todos los índices"
    assert not np.array_equal(te, np.arange(10)), "falta mezclar antes de cortar"
    tr2, va2, te2 = split_indices(100, 0.2, 0.1, seed=3)
    assert np.array_equal(te, te2), "con la misma semilla la partición debe repetirse"


check("split_indices", test_split_indices)

## 3. Sobreajuste, sesgo y varianza

Para ver el sobreajuste en un dibujo usamos datos inventados en los que sabemos la verdad: 20 puntos de la curva $\sin(2\pi x)$ con ruido. Les ajustamos polinomios de distinto grado con `np.polyfit` (la semana 4 lo usó con grado 1). El grado es el hiperparámetro: cuanto más alto, más flexible la curva.

In [ ]:
def true_curve(x):
    return np.sin(2 * np.pi * x)


x_tr = np.sort(rng.uniform(0, 1, 20))
t_tr = true_curve(x_tr) + rng.normal(0, 0.25, 20)
x_va = rng.uniform(0, 1, 200)
t_va = true_curve(x_va) + rng.normal(0, 0.25, 200)

grid = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, degree, title in zip(axes, [1, 3, 9], ["grado 1: muy rígido", "grado 3: justo", "grado 9: demasiado flexible"]):
    coefs = np.polyfit(x_tr, t_tr, degree)
    ax.plot(grid, true_curve(grid), color="gray", ls="--", label="la verdad")
    ax.plot(grid, np.polyval(coefs, grid), color="tab:orange", lw=2.5, label=f"polinomio de grado {degree}")
    ax.scatter(x_tr, t_tr, color="tab:blue", zorder=3, label="entrenamiento")
    ax.set_ylim(-1.8, 1.8)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

- **Grado 1, subajuste** (*underfitting*): una recta no puede seguir una onda. Falla en entrenamiento y en validación.
- **Grado 3**: sigue la forma de la verdad y deja pasar el ruido.
- **Grado 9, sobreajuste** (*overfitting*): pasa casi por cada punto, ruido incluido, y entre los puntos hace curvas locas. Es el estudiante que se memorizó las respuestas.

### Tu turno 3: la curva de error

Completa `mse_by_degree(x_tr, t_tr, x_va, t_va, degrees)`: para cada grado de `degrees`, ajusta un polinomio a los datos de entrenamiento con `np.polyfit` y calcula el MSE en entrenamiento y en validación (usa `np.polyval` para predecir). Devuelve dos arreglos de NumPy, `(train_mse, val_mse)`, con un valor por grado.

In [ ]:
def mse_by_degree(x_tr, t_tr, x_va, t_va, degrees):
    # TODO
    raise NotImplementedError

In [ ]:
def test_mse_by_degree():
    tr, va = mse_by_degree(x_tr, t_tr, x_va, t_va, [0, 1, 3])
    assert np.shape(tr) == (3,) and np.shape(va) == (3,), "debe haber un MSE por grado"
    assert np.isclose(tr[0], np.var(t_tr)), "con grado 0 (una constante) el MSE de entrenamiento es la varianza de t_tr"
    assert tr[2] < tr[1] < tr[0], "el error de entrenamiento baja al subir el grado"
    coefs = np.polyfit(x_tr, t_tr, 3)
    assert np.isclose(va[2], np.mean((np.polyval(coefs, x_va) - t_va) ** 2)), "revisa el MSE de validación"


check("mse_by_degree", test_mse_by_degree)

Con tu función dibujamos las dos curvas de error contra el grado:

In [ ]:
degrees = np.arange(0, 11)
try:
    train_mse, val_mse = mse_by_degree(x_tr, t_tr, x_va, t_va, degrees)
    best = degrees[np.argmin(val_mse)]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(degrees, train_mse, "o-", label="entrenamiento")
    ax.plot(degrees, val_mse, "o-", label="validación")
    ax.axvline(best, color="gray", ls=":", label=f"mejor grado en validación: {best}")
    ax.set_yscale("log")
    ax.set_xlabel("grado del polinomio (flexibilidad)")
    ax.set_ylabel("MSE (escala logarítmica)")
    ax.set_title("Subajuste a la izquierda, sobreajuste a la derecha")
    ax.legend()
    plt.show()
except NotImplementedError:
    print("Completa mse_by_degree en el Tu turno 3 para ver las curvas.")

Es el gráfico más importante de la clase. El error de entrenamiento **siempre** baja al dar más flexibilidad; el de validación baja, toca fondo y vuelve a subir. Se elige el punto más bajo de **validación**. Lo vas a ver todo el año con otra perilla en el eje horizontal: la profundidad del árbol, las épocas de entrenamiento de una red.

### Sesgo y varianza

¿Por qué falla cada extremo? Imagina que repetimos el experimento muchas veces, cada vez midiendo de nuevo los mismos 20 valores de $x$ (con ruido nuevo), y ajustamos el mismo grado. Miramos las curvas dentro del rango de los datos: fuera de él, un polinomio flexible se dispara.

- Con grado 1, las rectas salen casi iguales entre sí, pero todas lejos de la verdad: **sesgo** (*bias*) alto. Como un arquero con la mira torcida: sus flechas quedan juntas, pero lejos del centro.
- Con grado 9, cada conjunto de datos da una curva distinta: **varianza** (*variance*) alta. Un arquero con buena mira y pulso tembloroso: en promedio apunta al centro, pero cada flecha cae en otro lado.

In [ ]:
inside = np.linspace(x_tr.min(), x_tr.max(), 300)     # dentro del rango de los datos
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
print(f"{'grado':>6}{'sesgo²':>10}{'varianza':>11}")
for ax, degree in zip(axes, [1, 3, 9]):
    preds = []
    for _ in range(30):
        ts = true_curve(x_tr) + rng.normal(0, 0.25, 20)   # mismos x, ruido nuevo
        preds.append(np.polyval(np.polyfit(x_tr, ts, degree), inside))
        ax.plot(inside, preds[-1], color="tab:orange", alpha=0.25)
    preds = np.array(preds)                                    # (30, 300): 30 curvas
    bias2 = np.mean((preds.mean(axis=0) - true_curve(inside)) ** 2)
    variance = np.mean(preds.var(axis=0))
    print(f"{degree:>6}{bias2:>10.3f}{variance:>11.3f}")
    ax.plot(inside, true_curve(inside), color="black", lw=2, ls="--")
    ax.set_ylim(-1.8, 1.8)
    ax.set_title(f"grado {degree}: 30 mediciones distintas")
    ax.set_xlabel("x")
plt.tight_layout()
plt.show()

Al subir la flexibilidad, el sesgo baja y la varianza sube: es el **equilibrio sesgo-varianza** (*bias-variance trade-off*). El mejor modelo está en el medio. Más datos de entrenamiento bajan la varianza; por eso un modelo flexible necesita muchos ejemplos.

Lo mismo con el árbol en los datos de cáncer: la profundidad máxima (`max_depth`) es la perilla de flexibilidad. Ahora sí con tres partes: separamos la prueba y la dejamos guardada; del resto sacamos una validación.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(X_bc, y_bc, test_size=0.2, random_state=SEED, stratify=y_bc)
X_tr, X_val, y_tr, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=SEED, stratify=y_dev)
print(f"entrenamiento {len(y_tr)}, validación {len(y_val)}, prueba {len(y_test)} (guardada)")
print()
print(f"{'max_depth':>10}{'entrenamiento':>15}{'validación':>12}")
for depth in [1, 2, 3, 4, 5, 6, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(X_tr, y_tr)
    print(f"{str(depth):>10}{model.score(X_tr, y_tr):>15.3f}{model.score(X_val, y_val):>12.3f}")

El entrenamiento sube hasta 1.000, como esperábamos. Pero la validación salta de un lado a otro sin una forma clara: con solo 114 tumores de validación, un tumor más o menos bien clasificado mueve la *accuracy* casi un punto. ¿Podemos confiar en una sola partición para elegir? Eso resuelve la parte 4.

## 4. Validación cruzada

Primero midamos cuánto depende la nota de **qué** tumores tocaron en la validación. Mismo modelo, mismos datos, 20 particiones al azar distintas:

In [ ]:
scores = []
for seed in range(20):
    a_tr, a_val, b_tr, b_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=seed, stratify=y_dev)
    scores.append(DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(a_tr, b_tr).score(a_val, b_val))
print(f"accuracy de validación del mismo árbol: mínima {min(scores):.3f}, máxima {max(scores):.3f}")
plt.figure(figsize=(7, 2.5))
plt.plot(scores, np.zeros(20), "o", alpha=0.6, ms=10)
plt.yticks([])
plt.xlabel("accuracy de validación")
plt.title("Un mismo modelo, 20 particiones distintas")
plt.show()

Casi ocho puntos de diferencia solo por la suerte de la partición: más que la diferencia entre varios modelos que querríamos comparar. Juzgar a un futbolista por un solo partido es injusto; mejor mirar toda la temporada.

La **validación cruzada con $k$ pliegues** (*k-fold cross-validation*) hace eso:

1. Divide los datos de desarrollo en $k$ partes iguales (pliegues, *folds*).
2. Entrena $k$ veces: cada vez, un pliegue distinto hace de validación y los otros $k - 1$ de entrenamiento.
3. Reporta el promedio de los $k$ puntajes (y su desviación estándar, que dice cuánto varían).

Cada ejemplo se usa exactamente una vez para validar. Cuesta $k$ entrenamientos en vez de uno; con modelos clásicos eso es barato, con redes grandes a veces no se puede (y se usa una sola validación grande).

### Tu turno 4: los pliegues a mano

Completa `kfold_indices(n, k, seed)`: mezcla `0, …, n-1` con `np.random.default_rng(seed).permutation(n)`, corta la mezcla en `k` pedazos con `np.array_split` y devuelve una lista de `k` pares `(train_idx, val_idx)`: en el par `i`, la validación es el pedazo `i` y el entrenamiento es la unión de los demás (`np.concatenate`).

In [ ]:
def kfold_indices(n, k, seed):
    # TODO
    raise NotImplementedError

In [ ]:
def test_kfold_indices():
    folds = kfold_indices(23, 5, seed=1)
    assert len(folds) == 5, "debe haber k pares"
    val_all = np.concatenate([va for _, va in folds])
    assert sorted(val_all) == list(range(23)), "cada ejemplo debe validar exactamente una vez"
    for tr, va in folds:
        assert len(set(tr) & set(va)) == 0, "entrenamiento y validación no deben compartir ejemplos"
        assert len(tr) + len(va) == 23, "cada par debe cubrir todos los ejemplos"
    sizes = [len(va) for _, va in folds]
    assert max(sizes) - min(sizes) <= 1, f"los pliegues deben ser casi iguales: {sizes}"


check("kfold_indices", test_kfold_indices)

Con tus pliegues, la validación cruzada del árbol de profundidad 3 es un bucle de cinco líneas. Después, lo mismo con sklearn: `cross_val_score` hace el bucle, y `StratifiedKFold` además mantiene la proporción de malignos en cada pliegue.

In [ ]:
try:
    fold_scores = []
    for tr_idx, va_idx in kfold_indices(len(y_dev), 5, seed=SEED):
        model = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_dev[tr_idx], y_dev[tr_idx])
        fold_scores.append(model.score(X_dev[va_idx], y_dev[va_idx]))
    print("tus pliegues:   ", np.round(fold_scores, 3), f"→ promedio {np.mean(fold_scores):.3f}")
except NotImplementedError:
    print("Completa kfold_indices en el Tu turno 4 para correr tu propia validación cruzada.")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
sk_scores = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=SEED), X_dev, y_dev, cv=cv)
print("cross_val_score:", sk_scores.round(3), f"→ promedio {sk_scores.mean():.3f} ± {sk_scores.std():.3f}")

Ahora sí podemos elegir la profundidad del árbol con una curva confiable:

In [ ]:
depths = np.arange(1, 13)
cv_mean, cv_std, train_acc = [], [], []
for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    s = cross_val_score(model, X_dev, y_dev, cv=cv)
    cv_mean.append(s.mean())
    cv_std.append(s.std())
    train_acc.append(model.fit(X_dev, y_dev).score(X_dev, y_dev))
cv_mean, cv_std = np.array(cv_mean), np.array(cv_std)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, train_acc, "o-", label="entrenamiento")
ax.plot(depths, cv_mean, "o-", label="validación cruzada (promedio de 5)")
ax.fill_between(depths, cv_mean - cv_std, cv_mean + cv_std, alpha=0.2, color="tab:orange", label="± una desviación estándar")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.set_title("Elegir la profundidad del árbol")
ax.legend()
plt.show()
print("mejor profundidad según validación cruzada:", depths[np.argmax(cv_mean)])

La banda sombreada es ancha: de 3 a 6 las profundidades están empatadas dentro del ruido (los promedios difieren en menos de una centésima). Cuando dos opciones empatan, conviene la más simple, que tiene menos varianza: nos quedamos con `max_depth = 3`.

## 5. Métricas: *accuracy*, precisión, *recall*, F1

Piensa en una prueba de detección en un colegio de 1000 estudiantes, donde 10 tienen una enfermedad. Una "prueba" que dice **siempre "sano"** acierta con 990 de 1000: *accuracy* del 99%. Y no detecta a ningún enfermo. Cuando una clase es rara, la *accuracy* engaña.

Para ver mejor, contamos los cuatro casos posibles en la **matriz de confusión** (*confusion matrix*), tomando "enfermo" como la clase **positiva**:

| | predice positivo | predice negativo |
|---|---|---|
| **es positivo** | verdadero positivo (VP) | falso negativo (FN): se le escapó |
| **es negativo** | falso positivo (FP): falsa alarma | verdadero negativo (VN) |

Con una alarma de incendios: un falso positivo es que suene por una tostada quemada; un falso negativo, que no suene con el incendio.

- **Precisión** (*precision*) $= \dfrac{VP}{VP + FP}$: cuando la alarma suena, ¿qué fracción de las veces había fuego?
- ***Recall*** (sensibilidad) $= \dfrac{VP}{VP + FN}$: de todos los incendios, ¿qué fracción detectó?
- **F1** $= \dfrac{2 \cdot \text{precisión} \cdot \text{recall}}{\text{precisión} + \text{recall}}$: la media armónica de las dos. Es alta solo si **las dos** son altas.

Ejemplo con números: una prueba que marca 15 positivos, de los cuales 8 están enfermos, entre 10 enfermos en total. VP = 8, FP = 7, FN = 2. Precisión $8/15 = 0.53$, *recall* $8/10 = 0.80$, F1 $= 0.64$.

### Tu turno 5: las métricas a mano

Completa `precision_recall_f1(y_true, y_pred)` **sin bucles**, para etiquetas 0/1 con 1 = positivo. Cuenta VP, FP y FN con máscaras booleanas y `np.sum` (semana 2). Devuelve la tupla `(precision, recall, f1)`. Puedes suponer que hay al menos un positivo real y uno predicho.

In [ ]:
def precision_recall_f1(y_true, y_pred):
    # TODO
    raise NotImplementedError

In [ ]:
def test_precision_recall_f1():
    y_true = np.array([1] * 10 + [0] * 90)
    y_pred = np.array([1] * 8 + [0] * 2 + [1] * 7 + [0] * 83)
    p, r, f = precision_recall_f1(y_true, y_pred)
    assert np.isclose(p, 8 / 15) and np.isclose(r, 0.8), f"el ejemplo del texto: precisión {p:.3f}, recall {r:.3f}"
    yt, yp = rng.integers(0, 2, 200), rng.integers(0, 2, 200)
    got = precision_recall_f1(yt, yp)
    ref = precision_score(yt, yp), recall_score(yt, yp), f1_score(yt, yp)
    assert np.allclose(got, ref), f"no coincide con sklearn: {np.round(got, 3)} vs {np.round(ref, 3)}"


check("precision_recall_f1", test_precision_recall_f1)

De vuelta a los tumores, con maligno como clase positiva. Entrenamos una regresión logística con los datos de desarrollo y la comparamos con la línea base de la clase mayoritaria (siempre "benigno"). Aquí sí abrimos la prueba: es un ejemplo de clase; en un proyecto real se abre una sola vez, al final (parte 7).

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_dev, y_dev)
dummy = DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev)

for name, model in [("siempre benigno", dummy), ("regresión logística", logreg)]:
    pred = model.predict(X_test)
    print(f"{name}:")
    print("  matriz de confusión [[VN FP] [FN VP]]:", confusion_matrix(y_test, pred).tolist())
    print(f"  accuracy {accuracy_score(y_test, pred):.3f}   precisión {precision_score(y_test, pred, zero_division=0):.3f}"
          f"   recall {recall_score(y_test, pred):.3f}   F1 {f1_score(y_test, pred, zero_division=0):.3f}")

La línea base tiene 63% de *accuracy* y *recall* 0: no encuentra ni un tumor maligno. La regresión logística se equivoca en 3 de 114, y 2 de esos errores son tumores malignos que se le escaparon.

### El umbral: cambiar precisión por *recall*

`predict` dice "maligno" cuando la probabilidad pasa de 0.5. Ese **umbral** (*threshold*) es otra perilla. Bajarlo hace al modelo más desconfiado: atrapa más malignos (sube el *recall*) a cambio de más falsas alarmas (baja la precisión).

In [ ]:
proba = logreg.predict_proba(X_test)[:, 1]          # probabilidad de maligno
print(f"{'umbral':>7}{'FP':>5}{'FN':>5}{'precisión':>11}{'recall':>8}")
for th in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pred = (proba >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"{th:>7}{fp:>5}{fn:>5}{precision_score(y_test, pred):>11.3f}{recall_score(y_test, pred):>8.3f}")

¿Qué umbral usar? Depende del costo de cada error, y eso lo decide el problema, no el modelo:

- En una **detección de cáncer**, dejar pasar un tumor maligno (FN) es mucho peor que pedir una biopsia de más (FP): se prioriza el ***recall***, con un umbral bajo.
- En un **filtro de spam**, mandar a la papelera un correo importante (FP) es peor que dejar pasar un spam (FN): se prioriza la **precisión**.
- Si no hay una razón para preferir uno, F1 resume los dos.

En las competencias la métrica viene en el enunciado: leerla con cuidado es parte del problema (semana 30: "métrica mal leída" es un error típico).

## 6. Los cuatro modelos, lado a lado

Cada modelo tiene su propia forma de dividir el espacio. Los entrenamos con los pingüinos (dos medidas, así podemos dibujar) y pintamos la especie que predice cada modelo en cada punto del plano.

In [ ]:
models = {
    "regresión logística": make_pipeline(StandardScaler(), LogisticRegression()),
    "kNN, k = 1": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
    "kNN, k = 15": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "árbol, max_depth = 2": DecisionTreeClassifier(max_depth=2, random_state=SEED),
    "árbol sin límite": DecisionTreeClassifier(random_state=SEED),
}
species_names = np.unique(y_peng)
gx, gy = np.meshgrid(np.linspace(30, 62, 250), np.linspace(168, 235, 250))
fig, axes = plt.subplots(1, 5, figsize=(20, 4.2), sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_tr_peng, y_tr_peng)
    region = np.searchsorted(species_names, model.predict(np.c_[gx.ravel(), gy.ravel()])).reshape(gx.shape)
    ax.contourf(gx, gy, region, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#c6dbef", "#fdd0a2", "#c7e9c0"])
    for s, color in zip(species_names, ["tab:blue", "tab:orange", "tab:green"]):
        ax.scatter(*X_tr_peng[y_tr_peng == s].T, s=10, color=color, label=s)
    ax.set_title(f"{name}\nentren. {model.score(X_tr_peng, y_tr_peng):.3f} · prueba {model.score(X_te_peng, y_te_peng):.3f}", fontsize=10)
    ax.set_xlabel("largo del pico (mm)")
axes[0].set_ylabel("largo de la aleta (mm)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

- **Regresión logística**: fronteras rectas. Mucho sesgo si la frontera real es curva, poca varianza.
- **kNN**: fronteras con la forma que tengan los datos. Con $k = 1$ la frontera se dobla para alcanzar cada punto suelto (varianza alta); con $k = 15$ suaviza. **Más vecinos, menos flexible.**
- **Árbol**: cortes paralelos a los ejes, en escalones ("¿pico > 43 mm?"). Sin límite de profundidad, recorta franjas finas para atrapar puntos sueltos.

### Escalar importa (para kNN)

kNN mide distancias. En los datos de cáncer, el área del tumor va de 140 a 2500 y la suavidad de 0.05 a 0.16: sin escalar, la distancia la decide casi solo el área. Comparemos con validación cruzada:

In [ ]:
for name, model in [("kNN sin escalar", KNeighborsClassifier(n_neighbors=5)),
                    ("kNN con StandardScaler", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)))]:
    s = cross_val_score(model, X_dev, y_dev, cv=cv)
    print(f"{name:<24} accuracy {s.mean():.3f} ± {s.std():.3f}")

El *pipeline* además evita una trampa sutil: la **fuga de datos** (*data leakage*). El `StandardScaler` calcula medias y desviaciones; si lo ajustas con **todos** los datos antes de separar, la validación ya "vio" información de sí misma. Dentro de un *pipeline*, `cross_val_score` ajusta el escalador solo con los pliegues de entrenamiento de cada vuelta.

| Modelo | Idea | Perilla principal | ¿Escalar? | Para |
|---|---|---|---|---|
| `LinearRegression` | recta (o plano) que minimiza el MSE | — | no hace falta | regresión |
| `LogisticRegression` | capa lineal + *softmax* (semana 4) | `C` (regularización) | sí, ayuda | clasificación |
| `KNeighborsClassifier` | votan los $k$ vecinos más cercanos | `n_neighbors` | **sí, siempre** | clasificación (y regresión) |
| `DecisionTreeClassifier` | preguntas de sí o no en cadena | `max_depth` | no hace falta | clasificación (y regresión) |

## 7. La receta completa

Todo junto, en el orden en que lo vas a hacer en el Sprint 0 y en cada problema tabular de competencia:

1. Separar la prueba y no tocarla.
2. Una línea base tonta (`DummyClassifier`).
3. Comparar candidatos con validación cruzada sobre los datos de desarrollo.
4. Reentrenar el elegido con todos los datos de desarrollo.
5. Medir en la prueba **una sola vez** y reportar varias métricas, contra la línea base.

In [ ]:
candidates = {
    "siempre benigno": DummyClassifier(strategy="most_frequent"),
    "regresión logística": make_pipeline(StandardScaler(), LogisticRegression()),
    "kNN, k = 5": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "árbol, max_depth = 3": DecisionTreeClassifier(max_depth=3, random_state=SEED),
}
cv_f1 = {}
print(f"{'modelo':<22}{'F1 en validación cruzada':>26}")
for name, model in candidates.items():
    s = cross_val_score(model, X_dev, y_dev, cv=cv, scoring="f1")
    cv_f1[name] = s.mean()
    print(f"{name:<22}{s.mean():>18.3f} ± {s.std():.3f}")

best_name = max(cv_f1, key=cv_f1.get)
final = candidates[best_name].fit(X_dev, y_dev)
pred = final.predict(X_test)
print(f"\nelegido: {best_name}. En la prueba (una sola vez):")
print(f"  accuracy {accuracy_score(y_test, pred):.3f}   precisión {precision_score(y_test, pred):.3f}"
      f"   recall {recall_score(y_test, pred):.3f}   F1 {f1_score(y_test, pred):.3f}")

La línea base tiene F1 = 0 (no detecta ningún maligno), y por eso la comparación es útil: dice cuánto aporta el modelo sobre no hacer nada. Elegimos con F1 porque en este problema importan los malignos, no solo los aciertos totales.

## 8. Cierre

**Hoy vimos:**

- Aprendizaje supervisado: aprender de ejemplos con respuesta. Regresión (un número) o clasificación (una categoría). En sklearn, siempre `fit` y `predict`.
- Entrenamiento para ajustar, validación para elegir, prueba para la nota final, una sola vez. Mezclar y estratificar al separar.
- Sobreajuste: entrenamiento bien, datos nuevos mal. Subajuste: mal en los dos. La curva de validación en forma de U elige la flexibilidad. Sesgo (mira torcida) contra varianza (pulso tembloroso).
- Validación cruzada: $k$ pliegues, cada uno valida una vez; promedio y desviación estándar.
- *Accuracy* engaña con clases raras. Precisión: ¿las alarmas eran ciertas? *Recall*: ¿cuántos casos encontró? F1 junta las dos. El umbral las intercambia.
- Cuatro modelos: regresión lineal y logística (rectas), kNN (vecinos, hay que escalar), árbol (preguntas en escalones). Un *pipeline* evita la fuga de datos.

**Para la semana 6:**

1. **Sprint 0** (se publica al final de esta clase, asincrónico): limpiar un CSV desordenado, ajustar dos modelos de sklearn y reportar métricas contra una línea base, con la receta de la parte 7. Se guarda en tu repositorio de entregas en la carpeta `Sprint0/` y se entrega al inicio de la clase de la semana 6.
2. **Lectura de consulta para el Sprint 0** (MÓLÓ, en inglés): [Regression](https://colab.research.google.com/drive/1oAHb8vKEVqUfL-RVua3OHkDhNAPiABHA), [Decision Trees](https://colab.research.google.com/drive/1JCPwXKXioucTJigNkxCnuhKXL-h0gw-m) y [Non-Parametric Learning](https://colab.research.google.com/drive/1d7jCTYiRb9EMrYS79wQQzDUSjmFZy5vQ) (kNN).
3. **Cuota semanal** en [la plataforma](https://ioai.artix.tech/).

La semana 6 empieza el Bloque 2: del perceptrón a la red neuronal multicapa. La regresión logística de hoy es una red de una sola capa.